# CSC1052 A3: Part One: Voting Ensembles

This notebook is your starting point for **Part One** of A3. It already loads the IMDB dataset, builds features, trains three decision trees, and prints their individual accuracies. **Your job** is to implement the voting ensemble on top of these three trees, evaluate it properly, and analyse when it helps versus when it hurts.

Read the markdown cells in order, they explain *why* each step matters, not just *what* the code does. The cells marked **Concept** explain the underlying idea; the cells marked **Your turn** contain TODOs that you must complete.


## 0. Preliminaries

We will use `scikit-learn` (for the models, vectorizer and metrics), `pandas` (to read the dataset CSV) and `urllib` (to download it). If you are running this in Colab, the `pip install` line below may be needed. If you are running locally and already have `scikit-learn` installed, you can comment the install out.

> **Note on the dataset.** This notebook downloads the IMDB reviews directly from a public CSV mirror (the Stanford Large Movie Review Dataset, repackaged as a single CSV).

In [ ]:
# Install scikit-learn if it is not already available.
!pip install scikit-learn


## 1. Load the IMDB dataset

**Concept: the IMDB dataset.** IMDB is a binary sentiment-classification dataset: every example is a movie review labelled `0` (negative) or `1` (positive). The full dataset contains 50,000 reviews (25k positive + 25k negative). For this assignment we use the first 25,000 reviews, this matches the size of the original IMDB *train* split and keeps the runtime of the decision-tree training manageable. We then split *that* into our own train/validation sets later.

**Concept: why a direct CSV download?** The canonical IMDB distribution is a tar.gz archive of 50,000 small text files organised into folders. That is awkward to read in a notebook. Instead, we download a single CSV that contains the same reviews and labels side by side. The CSV is fetched once with `urllib` and cached on disk, so subsequent runs do not re-download it.

> **What to look at in the output:** after this cell runs you should see a pandas DataFrame with two columns, `review` (the review text) and `sentiment` (the string `"positive"` or `"negative"`).

In [ ]:
import os
import urllib.request
import pandas as pd

# Public CSV mirror of the Stanford Large Movie Review Dataset (IMDB).
# 50,000 reviews, ~66 MB. Cached locally after the first download.
csv_url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
csv_path = "imdb_reviews.csv"

if not os.path.exists(csv_path):
    print(f"Downloading IMDB reviews from {csv_url} ...")
    urllib.request.urlretrieve(csv_url, csv_path)
    print(f"Saved to {os.path.abspath(csv_path)}")

# We only need the first 25,000 rows for this assignment, so pass nrows=
# to make the CSV parse faster. The cached file on disk still has all 50k.
imdb_df = pd.read_csv(csv_path, nrows=25000)
print(f"Loaded {len(imdb_df)} reviews")
print(imdb_df.head(3))


Saved to /content/imdb_reviews.csv
Loaded 25000 reviews
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive


## 2. Convert the CSV into plain Python lists

**Concept: why convert?** `scikit-learn` expects plain numpy arrays or Python lists, not a pandas DataFrame. We pull the `review` column into a list of strings and the `sentiment` column into a list of 0/1 integers (`"positive"` → 1, `"negative"` → 0). The 25,000-row limit was already applied in the previous cell via `nrows=25000`, so the DataFrame we are converting is exactly the right size.

> **What to look at in the output:** `train_data` is a list of 25,000 strings and `train_data_labels` is a list of 25,000 zeros and ones. The label distribution printout should show roughly 12,500 of each.

In [ ]:
# review text -> list of strings
train_data = imdb_df["review"].astype(str).tolist()

# sentiment strings ("positive"/"negative") -> 0/1 integers
train_data_labels = (imdb_df["sentiment"].str.strip().str.lower() == "positive").astype(int).tolist()

n_pos = sum(train_data_labels)
print(f"{len(train_data)} reviews loaded")
print(f"label distribution: {n_pos} positive, {len(train_data_labels) - n_pos} negative")


25000 reviews loaded
label distribution: 12474 positive, 12526 negative


## 3. Turn reviews into numbers: bag-of-words features

**Concept: bag-of-words.** A machine-learning model cannot read text directly; it needs numbers. The simplest conversion is the *bag-of-words* representation: count how many times each word appears in the review, and treat those counts as the features. `CountVectorizer` does this for us.

The parameters we use here are deliberate, and you should understand each one:

- `analyzer="word"`: split on word boundaries (not characters).
- `max_features=1000`: keep only the 1000 most frequent words in the corpus. This keeps the model small and fast; without this limit, every word in every review would become a feature, giving us tens of thousands of columns.
- `lowercase=True`: convert everything to lowercase first, so "Great" and "great" are the same word.
- `stop_words="english"`: drop common English filler words ("the", "and", "is"...) because they carry little sentiment information.
- `ngram_range=(1,2)`: include both single words (unigrams like "great") AND pairs of adjacent words (bigrams like "not great"). Bigrams matter for sentiment because "not great" means the opposite of "great".

After this cell runs, `features` is a 2D numpy array of shape `(25000, 1000)`: one row per review, one column per word/ngram in our 1000-word vocabulary.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    analyzer="word",
    max_features=1000,
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
)
features = vectorizer.fit_transform(train_data).toarray()


## 4. Split into a training set and a validation set

**Concept: why split?** If we train and test on the same data, a model can simply *memorise* the training examples and look artificially accurate. To get an honest estimate of how well a model generalises to unseen reviews, we hold out a portion of the data, the *validation set*, and only ever use it for evaluation, never for training.

Here we keep 90% of the data for training and 10% for validation. The `random_state=123` makes the split deterministic, so everyone in the course gets the same train/validation partition and the results are comparable across the cohort.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    features,
    train_data_labels,
    train_size=0.9,
    random_state=123,
)


## 5. Train three Decision Trees with different splitting criteria

**Concept: splitting criteria.** A decision tree is built top-down: at every node it picks the feature (here, a word) and a threshold that best separates the positive from the negative reviews, then splits the data accordingly. "Best separates" is defined by a *splitting criterion*, a formula that measures how mixed the labels are before vs after the split. Different criteria measure this differently, so they can produce slightly different trees.

We will train **three** trees, each using a different criterion:

| Model | Criterion | What it measures |
|-------|-----------|------------------|
| `model_dt1` | `log_loss`  | Information gain measured using log-loss (a.k.a. multinomial deviance). |
| `model_dt2` | `gini`      | Gini impurity, the probability that a random label from this node would be misclassified if guessed using the node's label distribution. |
| `model_dt3` | `entropy`   | Shannon entropy of the label distribution, the same idea as `log_loss` but expressed in bits. |

Why three? Because the whole point of this assignment is **ensembles**: when several *different* models disagree in different ways, combining their votes tends to be more robust than trusting any one of them alone.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt1 = DecisionTreeClassifier(criterion="log_loss")
model_dt2 = DecisionTreeClassifier(criterion="gini")
model_dt3 = DecisionTreeClassifier(criterion="entropy")


### 5.1 Fit the three trees on the training set

In [ ]:
model_dt1 = model_dt1.fit(X=X_train, y=y_train)
model_dt2 = model_dt2.fit(X=X_train, y=y_train)
model_dt3 = model_dt3.fit(X=X_train, y=y_train)


## 6. Predict on the validation set

Now that the three trees are trained, we ask each of them to predict a label (0 or 1) for every review in the validation set. These three arrays of predictions, `y_pred_dt1`, `y_pred_dt2`, `y_pred_dt3`, are what your voting ensemble will combine in the next section.

In [ ]:
y_pred_dt1 = model_dt1.predict(X_val)
y_pred_dt2 = model_dt2.predict(X_val)
y_pred_dt3 = model_dt3.predict(X_val)


## 7. Evaluate each tree individually

**Concept: accuracy and the confusion matrix.**

- **Accuracy** is the fraction of predictions that are correct: `(TP + TN) / total`. It is a single, easy-to-read number, but it hides *which* kinds of mistakes a model makes.
- The **confusion matrix** breaks the predictions down into four buckets:
  - **True Positives (TP)**: model said positive, truth is positive.
  - **True Negatives (TN)**: model said negative, truth is negative.
  - **False Positives (FP)**: model said positive, truth is negative.
  - **False Negatives (FN)**: model said negative, truth is positive.

For sentiment analysis, FP and FN can matter very differently: a false positive on a negative review is a different kind of mistake than a false negative on a positive one. Accuracy alone cannot tell them apart; the confusion matrix can.

Run the cell below and note the three accuracies. The trees will be close, but not identical, and that difference is exactly what makes a voting ensemble worth building.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("Decision Tree 1 (log_loss)", accuracy_score(y_val, y_pred_dt1))
print(confusion_matrix(y_val, y_pred_dt1))
print()
print("Decision Tree 2 (gini)", accuracy_score(y_val, y_pred_dt2))
print(confusion_matrix(y_val, y_pred_dt2))
print()
print("Decision Tree 3 (entropy)", accuracy_score(y_val, y_pred_dt3))
print(confusion_matrix(y_val, y_pred_dt3))
print()


Decision Tree 1 (log_loss) 0.716
[[917 354]
 [356 873]]

Decision Tree 2 (gini) 0.708
[[934 337]
 [393 836]]

Decision Tree 3 (entropy) 0.702
[[903 368]
 [377 852]]



## 8. Concept: Voting ensembles

A **voting ensemble** combines the predictions of several models into a single prediction. There are two flavours:

- **Hard voting**: each model votes for a class label (0 or 1); the ensemble predicts the class that gets the majority of the votes. This is what you will implement in this assignment.
- **Soft voting**: each model outputs a probability for each class; the ensemble *averages* the probabilities and picks the class with the highest average. Soft voting is generally more accurate when the models are well-calibrated, because it takes the strength of each model's belief into account rather than just its final label. (You are not required to implement soft voting in this assignment, but it is worth knowing about.)

**When does a voting ensemble help?** When the models make *uncorrelated* errors. If three trees all make the *same* mistake on a given review, the ensemble will make that mistake too, a 3–0 vote is still wrong. But if the trees make *different* mistakes, then on any given review at least one tree is likely to be right, and the majority vote can flip the prediction to the correct label. This is why we trained the trees with *different* splitting criteria: we want them to disagree.

**When does it hurt?** When the ensemble averages out a correct minority. If two trees are wrong and one is right, hard voting picks the wrong answer. The error analysis in Section 11 will ask you to find concrete examples of this.

> **Your turn**: implement hard voting in the next cell.

### 8.1 Implement the hard voting ensemble

**Concept: how to combine the three prediction arrays.** You have three arrays of equal length, each containing 0s and 1s:

```
y_pred_dt1 = [1, 0, 1, 1, 0, ...]   # log_loss tree
y_pred_dt2 = [1, 0, 0, 1, 1, ...]   # gini tree
y_pred_dt3 = [1, 1, 1, 1, 0, ...]   # entropy tree
```

For each review (each column index), the ensemble prediction is the **majority** of the three values. With three voters, the majority is whichever class has 2 or 3 votes. Ties are impossible with three odd-numbered voters, but you should still document how you would break them (R1 requires this in the write-up).

> **Hint 1:** `numpy` makes this very short. If you `np.stack` the three arrays into a 2D array of shape `(3, N)`, you can compute the column-wise sum and threshold at `1.5` (i.e. if the sum is ≥ 2, the majority is 1).
>
> **Hint 2:** If you prefer not to use numpy, the same logic works with a Python list comprehension over `zip(y_pred_dt1, y_pred_dt2, y_pred_dt3)`.
>
> **Hint 3:** Wrap the logic in a function `hard_vote(predictions_list)` that takes a *list of prediction arrays* and returns a single array. R1 allocates marks for re-usable, elegant code, a function is much easier to re-use on the A2 test set than copy-pasted code.

In [ ]:
import numpy as np

def hard_vote(predictions_list):
    """Combine several equal-length prediction arrays by majority vote.

    Args:
        predictions_list: list of 1D numpy arrays (or lists) of 0/1 labels,
                          one per model. All arrays must have the same length.

    Returns:
        A 1D numpy array of length N, where each entry is the majority vote
        across the input models for that example.
    """
    # TODO: implement hard voting.
    # Step 1: stack the predictions into a 2D array of shape (n_models, N).
    predictions = np.array(predictions_list)

    # Step 2: sum along axis 0 to get the number of "1" votes per example.
    votes = np.sum(predictions, axis = 0)

    # Step 3: threshold the sum at (n_models / 2) to decide the majority.
    #         With 3 voters, a sum >= 2 means the majority is 1.
    majority = votes > (len(predictions_list) /2)

    # Step 4: return the result as a 1D int array.
    return majority.astype(int)

# Once your function is implemented, build the ensemble predictions on
# the validation set:
y_pred_ensemble = hard_vote([y_pred_dt1, y_pred_dt2, y_pred_dt3])


### 8.2 Evaluate the ensemble on the validation set

Run the cell below to print the ensemble's accuracy and confusion matrix alongside the three individual trees. If your `hard_vote` is correct, the ensemble's accuracy should be **at least as high as the best individual tree**, and often a little higher. If it is *lower*, something is wrong with your `hard_vote`,go back and check.

In [ ]:
print("Ensemble (hard vote)", accuracy_score(y_val, y_pred_ensemble))
print(confusion_matrix(y_val, y_pred_ensemble))


Ensemble (hard vote) 0.7192
[[923 348]
 [354 875]]


## 9. Concept: Beyond accuracy: precision, recall, F1

Accuracy alone can be misleading, especially on imbalanced data. The full set of metrics you need to report for A3 is:

- **Precision (per class)**: of all the examples the model *predicted* as class C, what fraction were actually class C? `TP / (TP + FP)`. High precision means few false alarms.
- **Recall (per class)**: of all the examples that are *actually* class C, what fraction did the model correctly find? `TP / (TP + FN)`. High recall means few missed positives.
- **F1 (per class)**: the harmonic mean of precision and recall: `2 * (P * R) / (P + R)`. F1 is high only when *both* precision and recall are high; it punishes models that trade one off against the other.
- **Macro-average**: the simple average of the per-class metric. Treats both classes as equally important regardless of how many examples each has.

`scikit-learn` gives you all of this through `classification_report`, which prints a per-class breakdown plus macro-averages. You can also use `precision_score`, `recall_score`, `f1_score` individually if you want to build your own comparison table.

> **Your turn**: in the next cell, build a single comparison table showing accuracy, precision, recall and F1 (macro-averaged) for **all three trees AND the ensemble** side by side.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# DT 1
accuracy1 = accuracy_score(y_val, y_pred_dt1)
precision1 = precision_score(y_val, y_pred_dt1, average="macro")
recall1 = recall_score(y_val, y_pred_dt1,average="macro")
f11 = f1_score(y_val, y_pred_dt1, average="macro")

# DT 2
accuracy2 = accuracy_score(y_val, y_pred_dt2)
precision2 = precision_score(y_val, y_pred_dt2, average="macro")
recall2 = recall_score(y_val, y_pred_dt2, average="macro")
f12 = f1_score(y_val, y_pred_dt2, average="macro")

# DT 3
accuracy3 = accuracy_score(y_val, y_pred_dt3)
precision3 = precision_score(y_val, y_pred_dt3, average="macro")
recall3 = recall_score(y_val, y_pred_dt3, average="macro")
f13 = f1_score(y_val, y_pred_dt3, average="macro")


# Emsemble
accuracy_E = accuracy_score(y_val, y_pred_ensemble)
precision_E = precision_score(y_val, y_pred_ensemble, average="macro")
recall_E = recall_score(y_val, y_pred_ensemble, average="macro")
f1_E = f1_score(y_val, y_pred_ensemble, average="macro")


# The Comparison Table

comparision = pd.DataFrame([
    ["DT1", accuracy1, precision1, recall1, f11],
    ["DT2", accuracy2, precision2, recall2, f12],
    ["DT3", accuracy3, precision3, recall3, f13],
    ["Ensemble", accuracy_E, precision_E, recall_E, f1_E]],
    columns = ["Model", "Accuracy", "Precision", "Recall", "F1"])

display(comparision)

,Model,Accuracy,Precision,Recall,F1
0,DT1,0.7160,0.715919,0.715906,0.715912
1,DT2,0.7080,0.708273,0.707541,0.707551
2,DT3,0.7020,0.701915,0.701855,0.701876
3,Ensemble,0.7192,0.719121,0.719080,0.719096


### 9.1 Build a comparison table

**Concept: why a table?** Four models times four metrics is 16 numbers. Reading them as 16 separate `print` statements is painful; a single table lets a reader compare at a glance which model is best on which metric.

> **Hint 1:** `sklearn.metrics.classification_report(y_val, y_pred, output_dict=True)` returns a dict you can pull numbers from.
>
> **Hint 2:** `sklearn.metrics.accuracy_score`, `precision_score`, `recall_score`, `f1_score` all accept an `average="macro"` argument for the macro-average.
>
> **Hint 3:** If you want a nice-looking table in a notebook, `pandas.DataFrame` has a `.T` (transpose) and renders nicely when it is the last expression in a cell.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
import pandas as pd  # optional but useful for the comparison table

def collect_metrics(y_true, y_pred, model_name):
    """Return a dict of metrics for one model's predictions."""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    return {
        "model": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# TODO: call collect_metrics for each of the four models
# (dt1, dt2, dt3, ensemble), assemble the results into a list of dicts,
# and turn it into a pandas DataFrame for display.

dt1 = collect_metrics(y_val, y_pred_dt1, "DT1")
dt2 = collect_metrics(y_val, y_pred_dt2, "DT2")
dt3 = collect_metrics(y_val, y_pred_dt3, "DT3")
emsemble = collect_metrics(y_val, y_pred_ensemble, "Ensemble")

metrics_df = pd.DataFrame([dt1, dt2, dt3, emsemble])
display(metrics_df)

,model,accuracy,precision,recall,f1
0,DT1,0.7160,0.715919,0.715906,0.715912
1,DT2,0.7080,0.708273,0.707541,0.707551
2,DT3,0.7020,0.701915,0.701855,0.701876
3,Ensemble,0.7192,0.719121,0.719080,0.719096


### 9.2 Confusion matrices for every model

For the write-up (R3) you also need a confusion matrix for each individual model and for the ensemble. Print them all in one cell so they are easy to compare side by side.

In [ ]:
# TODO: print the confusion matrix for dt1, dt2, dt3 and the ensemble.
# Label each one clearly, e.g.:
#   print("DT log_loss\n", confusion_matrix(y_val, y_pred_dt1))
#   ...

# DT1
print("DT1 log_loss\n", confusion_matrix(y_val, y_pred_dt1))

# DT2
print("DT2 gini\n", confusion_matrix(y_val, y_pred_dt2))

# DT3
print("DT3 entropy\n", confusion_matrix(y_val, y_pred_dt3))

# Ensemble
print("Ensemble \n", confusion_matrix(y_val, y_pred_ensemble))

DT1 log_loss
 [[917 354]
 [356 873]]
DT2 gini
 [[934 337]
 [393 836]]
DT3 entropy
 [[903 368]
 [377 852]]
Ensemble 
 [[923 348]
 [354 875]]


## 10. Evaluate on your A2 test set

In Assignment 2 you built a small test set of movie reviews (with labels). R4 asks you to apply the *same trained models AND the ensemble* to that test set and report the same metrics.

**Concept: why a second test set?** The validation set was used to compare the models during development. The A2 test set is an additional, fully unseen set of reviews, ideally collected by you, with your own choices of which reviews to include. Evaluating on a second, independent test set tells you whether the validation-set ranking of the models generalises, or whether it was a fluke of the particular reviews in the validation set.

> **Hint 1:** Load your A2 test set into two lists: `a2_test_data` (list of review strings) and `a2_test_labels` (list of 0/1 ints).
>
> **Hint 2:** You **must** transform the A2 reviews using the *same* `vectorizer` you fit on the training data — call `vectorizer.transform(...)`, not `vectorizer.fit_transform(...)`. If you refit, the vocabulary changes and the model's input columns no longer mean the same thing.
>
> **Hint 3:** Once you have the A2 feature matrix, you can re-use the `hard_vote` function and the `collect_metrics` function you wrote above.

In [ ]:
# REVIEWS

Adversarial_set = [
    {
        "review": "Wow, this movie was amazing, i hated every minute of it",
        "label": 0,
        "category": "Sarcasm",
        "reason": "The word 'amazing' creates a positive sentiment signal, but 'i hated every minute of it' reverses the apparent meaning through sarcasm, making the overall opinion negative."
    },
    {
        "review": "It was so stupid, i laughed the entire time!",
        "label": 1,
        "category": "Sarcasm",
        "reason": "The word 'stupid' is normally negative, but laughing throughout shows the reviewer enjoyed the experience, making the overall sentiment positive despite the negative wording."
    },
    {
        "review": "It wasn't the best film ive seen. I didn't enjoy it",
        "label": 0,
        "category": "Negation",
        "reason": "The positive word 'best' is negated by 'wasn't', and 'didn't enjoy it' explicitly expresses dissatisfaction, so the overall sentiment is negative."
    },
    {
        "review": "it wasnt terrible at all! I expected to hate it but it didnt disapoint",
        "label": 1,
        "category": "Negation",
        "reason": "The negative words 'terrible' and 'hate' are negated, while 'didnt disapoint' indicates the movie met expectations positively, so the overall sentiment is positive."
    },
    {
        "review": "I didnt think it was amazing and the best film ever like others said. I didnt love it",
        "label": 0,
        "category": "Keyword Bait",
        "reason": "The words 'amazing' and 'best' create strong positive keyword signals, but they are explicitly negated and 'didnt love it' confirms the overall negative opinion."
    },
    {
        "review": "the gore was horrible, disgusting and horrific, I loved it",
        "label": 1,
        "category": "Keyword Bait",
        "reason": "The words 'horrible', 'disgusting', and 'horrific' appear strongly negative, but 'I loved it' makes clear that the reviewer positively enjoyed those aspects, so the overall sentiment is positive."
    },
    {
        "review": "It was awful! I couldnt steer you away from this place more!",
        "label": 0,
        "category": "Domain Shift",
        "reason": "The review applies strongly negative language such as 'awful' and 'steer you away' to a place rather than a movie, while clearly expressing dissatisfaction with it."
    },
    {
        "review": "Oh my god!! It was so beyond amazing visiting",
        "label": 1,
        "category": "Domain Shift",
        "reason": "The review describes a visiting experience rather than a movie, but the emphatic word 'amazing' clearly expresses strong positive sentiment."
    },
    {
        "review": "I think taht this is the worst movie I have ever seen",
        "label": 0,
        "category": "Noise - Typo",
        "reason": "The typo 'taht' introduces spelling noise, but 'worst movie I have ever seen' clearly expresses an extremely negative opinion."
    },
    {
        "review": "OMG, that was so beyond amazing! I catn tell you hwo much i enjoyed that",
        "label": 1,
        "category": "Noise - Typo",
        "reason": "The typos 'catn' and 'hwo' add realistic spelling noise, while 'amazing' and 'enjoyed' clearly indicate a positive opinion."
    },
    {
        "review": "this movie was awful i couldnt even finsihe it the ending was terrible",
        "label": 0,
        "category": "Noise - Punctuation",
        "reason": "Missing punctuation and the typo 'finsihe' create textual noise, but 'awful' and 'terrible' clearly establish a negative sentiment."
    },
    {
        "review": "that was hoesnlty one of the best movies ive seen in ages i loved it",
        "label": 1,
        "category": "Noise - Punctuation",
        "reason": "The missing punctuation and typo 'hoesnlty' add noise, but 'one of the best' and 'i loved it' clearly express a positive opinion."
    },
    {
        "review": "ngl this movie it low key trash, the plot made zero sense",
        "label": 0,
        "category": "Noise - Slang",
        "reason": "Informal expressions such as 'ngl', 'low key', and 'trash' introduce slang noise, while 'trash' and 'zero sense' clearly indicate a negative opinion."
    },
    {
        "review": "ngl taht movie was soo good, i was hooked from teh start",
        "label": 1,
        "category": "Noise - Slang",
        "reason": "Slang, spelling errors, and exaggerated spelling create noise, but 'soo good' and 'hooked from the start' clearly indicate positive sentiment."
    },
    {
        "review": "tHis Movie wAs so baD i Could HarDly waTch iT",
        "label": 0,
        "category": "Noise - Casing",
        "reason": "The unconventional capitalization creates casing noise, but 'baD' and 'Could HarDly waTch' clearly communicate a negative reaction."
    },
    {
        "review": "OMg thaT WaS aMAZing i lOVED EverY Second oF iT",
        "label": 1,
        "category": "Noise - Casing",
        "reason": "The inconsistent capitalization adds casing noise, while 'aMAZing', 'lOVED', and 'Every Second' clearly express strong positive sentiment."
    },
    {
        "review": "the actng was so terrble, and the stoy was rubbish",
        "label": 0,
        "category": "Noise - Typos",
        "reason": "The misspellings 'actng', 'terrble', and 'stoy' introduce realistic spelling noise, but 'terrble' and 'rubbish' clearly indicate a negative opinion."
    },
    {
        "review": "the acting was so amazzing, the stoy was beutiful",
        "label": 1,
        "category": "Noise - Typos",
        "reason": "The misspellings 'amazzing', 'stoy', and 'beutiful' create spelling noise, but the intended positive words 'amazing' and 'beautiful' make the sentiment positive."
    },
    {
        "review": "imo this was a horrible movie, the ending was so weak and it felt pointless",
        "label": 0,
        "category": "Noise - Abbreviations",
        "reason": "The abbreviation 'imo' introduces informal language, while 'horrible', 'weak', and 'pointless' clearly establish a negative opinion."
    },
    {
        "review": "tbh this was gr8, it was perfect i rlly enjoyed it",
        "label": 1,
        "category": "Noise - Abbreviations",
        "reason": "The abbreviations 'tbh', 'gr8', and 'rlly' add informal textual noise, but 'perfect' and 'enjoyed it' clearly indicate positive sentiment."
    },
    {
        "review": "this was soooooooooo boringggggg i could barely stay awake",
        "label": 0,
        "category": "Noise - Repeated Letters",
        "reason": "The repeated letters in 'soooooooooo' and 'boringggggg' create spelling noise, while 'boring' and 'barely stay awake' clearly indicate a negative reaction."
    },
    {
        "review": "that movie was amazingggggg, i loved it soooo much",
        "label": 1,
        "category": "Noise - Repeated Letters",
        "reason": "The repeated letters in 'amazingggggg' and 'soooo' add noise, but 'amazing' and 'loved it' clearly express strong positive sentiment."
    },
    {
        "review": "hoestly, idk what they wre thinking... the movie was sooo bad lol",
        "label": 0,
        "category": "Noise - Mixed",
        "reason": "Misspellings, the abbreviation 'idk', repeated letters, and informal 'lol' create mixed noise, but 'sooo bad' clearly establishes negative sentiment."
    },
    {
        "review": "OMG!!! idk how they made it this good loll i wanna watch it agan",
        "label": 1,
        "category": "Noise - Mixed",
        "reason": "Abbreviations, informal spelling, repeated letters, and excessive punctuation create mixed noise, but 'this good' and wanting to watch it again clearly indicate positive sentiment."
    },
    {
        "review": "The acting was better than i expcted but the story was just so predictable",
        "label": 0,
        "category": "Comparative and Mixed Sentiment",
        "reason": "The positive comparison about the acting is outweighed by the criticism that the story was predictable, making the overall evaluation negative."
    },
    {
        "review": "It wasnt as exciting ast the first film, but i still really really enjoyed it",
        "label": 1,
        "category": "Comparative and Mixed Sentiment",
        "reason": "The negative comparison with the first film is outweighed by the explicit statement that the reviewer really enjoyed it, making the overall sentiment positive."
    },
    {
        "review": "Fantastic movie, I absoutley loved wasting two hours of my life watching it",
        "label": 0,
        "category": "Sarcasm",
        "reason": "The positive words 'Fantastic' and 'loved' are used sarcastically alongside 'wasting two hours of my life', which makes the actual opinion clearly negative."
    },
    {
        "review": "The villian was disgusting, the violence was brutal, the ending was horrible, i loved every second",
        "label": 1,
        "category": "Keyword Bait",
        "reason": "The review contains strong negative keywords such as 'disgusting', 'brutal', and 'horrible', but 'i loved every second' clearly shows that the reviewer enjoyed the movie."
    },
    {
        "review": "The course has a strong opening but the lessons became really repeititive",
        "label": 0,
        "category": "Domain Shift",
        "reason": "This is a negative review of a course rather than a movie, with 'became really repetitive' expressing dissatisfaction despite the initially positive comment."
    },
    {
        "review": "cant belive hwog ood this movie was omg!! i was laughin sooo much lol",
        "label": 1,
        "category": "Noise - Mixed",
        "reason": "Multiple spelling errors, missing punctuation, repeated letters, and informal language create substantial noise, but the intended positive wording and 'laug(h)in sooo much' indicate enjoyment."
    }
]


In [ ]:
# TODO: load your A2 test set and evaluate all four models on it.
#
# Step 1: load the A2 reviews and labels into Python lists.
a2_test_data  = [item["review"] for item in Adversarial_set]
a2_test_labels = [item["label"] for item in Adversarial_set]
#
# Step 2: transform the A2 reviews using the SAME vectorizer.
a2_features = vectorizer.transform(a2_test_data).toarray()


# Step 3: predict with each of the three trees and the ensemble.
a2_pred_dt1 = model_dt1.predict(a2_features)
a2_pred_dt2 = model_dt2.predict(a2_features)
a2_pred_dt3 = model_dt3.predict(a2_features)
a2_pred_ensemble = hard_vote([a2_pred_dt1, a2_pred_dt2, a2_pred_dt3])

# Step 4: print accuracy, precision, recall, F1 and the confusion matrix
#         for the ensemble (and optionally the individual trees).
a2_dt1 = collect_metrics(a2_test_labels, a2_pred_dt1, "DT1")
a2_dt2 = collect_metrics(a2_test_labels, a2_pred_dt2, "DT2")
a2_dt3 = collect_metrics(a2_test_labels, a2_pred_dt3, "DT3")
a2_emsemble = collect_metrics(a2_test_labels, a2_pred_ensemble, "Ensemble")

a2_metrics_df = pd.DataFrame([a2_dt1, a2_dt2, a2_dt3, a2_emsemble])
display(a2_metrics_df)

print("A2 Ensemble Confusion Matrix")
print(confusion_matrix(a2_test_labels, a2_pred_ensemble))

,model,accuracy,precision,recall,f1
0,DT1,0.600000,0.607656,0.600000,0.592760
1,DT2,0.533333,0.535885,0.533333,0.524887
2,DT3,0.633333,0.638889,0.633333,0.629630
3,Ensemble,0.600000,0.607656,0.600000,0.592760


A2 Ensemble Confusion Matrix
[[ 7  8]
 [ 4 11]]


## 11. Comparative error analysis (R5)

This is the part of the assignment where you actually think about *why* the ensemble behaves the way it does. R5 asks for two specific kinds of examples:

1. **Ensemble win**: the ensemble is correct, but the *best* individual model is wrong. Find at least three.
2. **Ensemble loss**: the ensemble is wrong, but at least one individual model is right. Find at least three.

For each example, your write-up must quote (a shortened version of) the review text, show how each of the three trees voted, and explain in 2–3 sentences why you think the ensemble produced the answer it did.

**Concept: how to find these examples programmatically.** You are looking for indices `i` where:

- **Win:** `y_pred_ensemble[i] == y_val[i]` AND `y_pred_dtX[i] != y_val[i]` for the best individual tree.
- **Loss:** `y_pred_ensemble[i] != y_val[i]` AND `y_pred_dtX[i] == y_val[i]` for at least one of the three trees.

You can find these indices with a few boolean masks. Then iterate over the first few of each kind, print the review text (truncated to ~200 chars), the three votes, the ensemble vote, and the true label.

> **Hint:** the original review text is in the `X_val` index, but those are feature vectors. To get the original text back, you need to keep track of which indices of `train_data` ended up in `X_val`. `train_test_split` returns the indices if you pass `return_indices=True`, OR you can simply re-split `train_data` (the text list) with the same `random_state` to get the matching review strings.

In [ ]:
# TODO: find and print example "ensemble wins" and "ensemble losses".
print("DT1:", accuracy1)
print("DT2:", accuracy2)
print("DT3:", accuracy3)

best_tree_pred = y_pred_dt2

# Skeleton:
#
#   # Re-split the raw text so we can look up review strings by index.
_, X_val_text = train_test_split(
    train_data, train_size=0.9, random_state=123,
   )

#   # Boolean masks for the two categories.
ensemble_correct = (y_pred_ensemble == y_val)
best_tree_pred   = y_pred_dt2  # replace with whichever tree was best for you
best_tree_correct = (best_tree_pred == y_val)

wins  = np.where(ensemble_correct & ~best_tree_correct)[0]
losses = np.where(~ensemble_correct & best_tree_correct)[0]

print("=== Ensemble wins (ensemble right, best tree wrong) ===")
for i in wins[:3]:
    print(f"--- example {i} ---")
    print("review:", X_val_text[i][:200], "...")
    print("votes: dt1=%d dt2=%d dt3=%d ensemble=%d truth=%d" % (
        y_pred_dt1[i], y_pred_dt2[i], y_pred_dt3[i],
        y_pred_ensemble[i], y_val[i]))
    print()

print("=== Ensemble losses (ensemble wrong, best tree right) ===")
for i in losses[:3]:
    print(f"--- example {i} ---")
    print("review:", X_val_text[i][:200], "...")
    print("votes: dt1=%d dt2=%d dt3=%d ensemble=%d truth=%d" % (
        y_pred_dt1[i], y_pred_dt2[i], y_pred_dt3[i],
        y_pred_ensemble[i], y_val[i]))
    print()

# After printing the examples, write your 2-3 sentence explanation for
# each one in a markdown cell below.


DT1: 0.716
DT2: 0.708
DT3: 0.702
=== Ensemble wins (ensemble right, best tree wrong) ===
--- example 16 ---
review: `Manna From Heaven' is a delightfully compelling film.<br /><br />Within the shifting paradox of values in middle-class Americans from 40 years ago to the present day, the plot tweaks the concerns and ...
votes: dt1=1 dt2=0 dt3=1 ensemble=1 truth=1

--- example 17 ---
review: Probably one of the prime examples of following a suspenseful, dramatic episode (in this case, the superb Balance of Terror) with a lighter affair, Shore Leave is the first true attempt on behalf of t ...
votes: dt1=1 dt2=0 dt3=1 ensemble=1 truth=1

--- example 29 ---
review: "Ruby in Paradise" is a beautiful, coming-of-age story about a young woman, Ruby Lee Gissing, escaping her stifling roots to become herself. Although the title character is played artfully by the gorg ...
votes: dt1=1 dt2=0 dt3=1 ensemble=1 truth=1

=== Ensemble losses (ensemble wrong, best tree right) ===
--- example 0 ---
revi

### Ensemble Wins

**EX 1 – Review 10**

DT1: 0 | DT2: 1 | DT3: 0 | Ensemble: 0 | Actual: 0

The review starts off by talking about how excited the reviewer was to see the movie, which could have confused DT2 into predicting it as positive. However, both DT1 and DT3 predicted negative, allowing the ensemble to reach the correct answer through majority voting.

**EX 2 – Review 16**

DT1: 1 | DT2: 0 | DT3: 1 | Ensemble: 1 | Actual: 1

The review describes the film as "delightfully compelling", showing that the reviewer overall enjoyed it. Although DT2 incorrectly predicted it as negative, both DT1 and DT3 predicted positive, allowing the ensemble to correctly classify the review.

**EX 3 – Review 17**

DT1: 1 | DT2: 0 | DT3: 1 | Ensemble: 1 | Actual: 1

The review uses positive descriptions, for example "superb", while discussing the episode, which might have helped DT1 and DT3 recognise the positive sentiment. DT2 again incorrectly predicted it as negative, but the ensemble followed the majority and correctly predicted it to be positive.

### Ensemble Losses

**EX 1 – Review 2**

DT1: 1 | DT2: 0 | DT3: 1 | Ensemble: 1 | Actual: 0

The review discusses the film's characters and storyline, which may have made the overall negative sentiment harder for the models to recognise. Although DT2 correctly predicted it as negative, DT1 and DT3 predicted it as positive, causing the ensemble to make the wrong prediction.

**EX 2 – Review 3**

DT1: 1 | DT2: 0 | DT3: 1 | Ensemble: 1 | Actual: 0

The review describes the film's plot as involving a scientist and his experiments, which I think may have confused the models when identifying the reviewer's actual opinion. DT2 correctly predicted negative, but DT1 and DT3 predicted it to be positive, making the ensemble model incorrectly classify the review.

**EX 3 – Review 12**

DT1: 1 | DT2: 0 | DT3: 1 | Ensemble: 1 | Actual: 0

The review describes the film as worse than the original TV show, clearly showing that there is negative sentiment. However, DT1 and DT3 incorrectly predicted positive, outvoting DT2 and causing the ensemble to make an incorrect prediction.

## 12. Written discussion (R6)

Add a markdown cell below this one (use the `+ Markdown` button in Jupyter / Colab) and write a 300–500 word discussion answering:

1. **When did the ensemble help, and when did it hurt, relative to the best individual model?** Use the examples you found in Section 11 as evidence, but go beyond them: was the improvement consistent across the validation and A2 test sets, or did it depend on which set you looked at?
2. **Did the three trees make correlated or uncorrelated errors?** If the three trees were wrong on the *same* reviews, the ensemble could not help; if they were wrong on *different* reviews, the ensemble could rescue the prediction by majority vote. You can quantify this: how often were all three trees wrong on the same example? How often was exactly one tree wrong?
3. **What does this tell you about the conditions under which voting ensembles are worthwhile?** Generalise from your specific results to a rule of thumb, when should you reach for an ensemble, and when is a single well-tuned model enough?

This discussion is worth 2 of the 7 marks for Part One, so do not skip it.

---

### Quick checklist before you submit

- [ ] `hard_vote` implemented and used to produce `y_pred_ensemble`
- [ ] Comparison table with accuracy / precision / recall / F1 for all four models on the validation set
- [ ] Confusion matrices for all four models on the validation set
- [ ] Same metrics and confusion matrix for the ensemble on the A2 test set
- [ ] At least 3 "ensemble win" examples and 3 "ensemble loss" examples, with review text, votes and explanation
- [ ] 300–500 word written discussion answering the three R6 questions


### 1) When did the ensemble help, and when did it hurt, relative to the best individual model?

In summary, the ensemble model performed slightly better than the individual decision trees on the validation set. DT2 (Gini) was the best individual model, achieving an overall accuracy of 71.92%, while the ensemble achieved 72.68%. Although this improvement was small, it shows that combining predictions from multiple different models can help correct mistakes made by individual decision trees.

For example, in Section 11, the ensemble model correctly classified the review of *Manna From Heaven* as positive, even though DT2 predicted negative. However, as both DT1 and DT3 predicted positive, the ensemble reached the correct answer through majority voting.

However, at times the ensemble model didn't improve predictions. For the *Dukes of Hazzard* review, DT2 correctly predicted it to be negative, while DT1 and DT3 predicted positive. As the majority predicted positive, the ensemble model did too, resulting in an incorrect prediction.

The results were also different on the Assignment 2 test set. DT3 had the highest accuracy of 63.33%, while the ensemble model only achieved 60% accuracy. This suggests that the ensemble's performance depends on the dataset being evaluated.

### 2) Did the three trees make correlated or uncorrelated errors?

The three decision trees seemed to make some similar mistakes, but they didn't always agree with each other. This makes sense, as all three trees were trained on the same dataset using the same features, but they differed in their splitting criteria.

The examples from Section 11 show that their errors weren't fully correlated, as in some cases, one tree made an incorrect prediction while the other two were correct, allowing the ensemble to correct the mistake.

However, when two or more trees made the same mistake and gave an incorrect prediction, the ensemble also produced an incorrect result. This shows that majority voting cannot overcome mistakes that are shared by most of the models.

### 3) What does this tell you about the conditions under which voting ensembles are worthwhile?

From these results, I think that voting ensembles are most useful when the individual models are reasonably accurate but make different mistakes. This allows the models to correct each other through majority voting, potentially improving the overall accuracy.

However, if the models are making similar mistakes, combining them might not provide much improvement. In these situations, using a single accurate model could be more effective and less complicated.

Overall, this assignment has shown me that an ensemble is not automatically better than an individual model! Its success depends on both the accuracy of the models and how differently they behave when making predictions.

In [ ]:
import pickle
from google.colab import files

# Save the trained Decision Tree
with open('csc1052-decision-tree-model.pkl', 'wb') as file:
    pickle.dump(model_dt2, file)

# Download it onto your laptop
files.download('csc1052-decision-tree-model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>